In [1]:
import torch
import torch.nn as nn

# [오늘의 목표] 정보 손실을 막는 주파수 전용 보관함 설계
class ResidualFrequencyBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualFrequencyBlock, self).__init__()

        # 1. 정보를 요약하는 돋보기 (Convolution)
        self.conv_path = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), # 깔끔하게 정리
            nn.ReLU(),                    # 중요한 특징만 남기기
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels)
        )

        # 2. 정보 유실 방지 지름길 (Residual Connection)
        # 들어온 채널과 나가는 채널이 다를 때만 크기를 맞춰주는 통로를 만듭니다.
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        # 공부한 내용(conv_path)에 원래 내용(shortcut)을 더하기(+) 합니다!
        # 이게 바로 정보를 안 까먹게 해주는 '레지듀얼'의 마법입니다.
        out = self.conv_path(x)
        out += self.shortcut(x)

        return torch.relu(out)

# [확인 작업] 잘 만들어졌는지 가짜 데이터를 넣어봅니다.
if __name__ == "__main__":
    # 배치 1개, 9채널(L1 결과물), 112x112 크기의 가짜 사진
    sample_input = torch.randn(1, 9, 112, 112)

    # 보관함 생성 (9채널을 넣어서 36채널로 요약하기)
    block = ResidualFrequencyBlock(9, 36)
    output = block(sample_input)

    print(f"입력 크기: {sample_input.shape}")
    print(f"출력 크기: {output.shape}") # (1, 36, 112, 112)가 나오면 성공!

입력 크기: torch.Size([1, 9, 112, 112])
출력 크기: torch.Size([1, 36, 112, 112])


In [2]:
class FrequencyBranch(nn.Module):
    def __init__(self):
        super().__init__()
        # 1. 보관함 2개를 준비합니다.
        self.block1 = ResidualFrequencyBlock(9, 18)
        self.block2 = ResidualFrequencyBlock(18, 36)
        # 2. 크기를 28x28로 줄여주는 요술 주머니
        self.pool = nn.AdaptiveAvgPool2d((28, 28))

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.pool(x)

In [3]:
class FrequencyFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        # 1. 아까 만든 L1, L2 연구소 두 개를 가져옵니다.
        self.l1_branch = FrequencyBranch()
        self.l2_branch = FrequencyBranch()

        # 2. 마지막으로 72장의 보고서를 144장으로 요약해줄 최종 돋보기
        # 28x28 크기를 14x14로 줄이기 위해 stride=2를 씁니다.
        self.final_conv = nn.Sequential(
            nn.Conv2d(72, 144, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(144),
            nn.ReLU()
        )

    def forward(self, l1_input, l2_input):
        # 각 연구소에서 보고서를 받아옵니다.
        feat1 = self.l1_branch(l1_input) # (B, 36, 28, 28)
        feat2 = self.l2_branch(l2_input) # (B, 36, 28, 28)

        # [합치기] 36 + 36 = 72채널로 만듭니다.
        combined = torch.cat([feat1, feat2], dim=1) # (B, 72, 28, 28)

        # [최종 요약] 144채널, 14x14 크기로 변신!
        final_out = self.final_conv(combined)
        return final_out

# [최종 확인 테스트]
if __name__ == "__main__":
    model = FrequencyFeatureExtractor()

    # 가짜 데이터 넣기 (한결 씨가 DWT로 뽑은 9채널 데이터들)
    mock_l1 = torch.randn(1, 9, 112, 112)
    mock_l2 = torch.randn(1, 9, 56, 56)

    output = model(mock_l1, mock_l2)
    print(f"최종 결과물 크기: {output.shape}")
    # [1, 144, 14, 14] 가 나오면 오늘 업무 완벽 종료입니다!

최종 결과물 크기: torch.Size([1, 144, 14, 14])
